# Tab-pretext split v2 — MORPH model only (resume after OOM)
`tab-mdn-color-v1` finished before the OOM; its eval + embedding were done at home.
This notebook trains only the remaining **MORPH** model (`conc_r`, `log_petroR90_r`),
evaluates it, and uploads its embedding. One training per runtime = no OOM.

Runtime → Change runtime type → **GPU**.


In [ ]:
import os
os.environ['CUTOUT_SIZE'] = '24'
![ -d /content/Photo-Z-CNN-Model ] || git clone -b main https://github.com/zhhrozhh/Photo-Z-CNN-Model.git /content/Photo-Z-CNN-Model
%cd /content/Photo-Z-CNN-Model
!pip -q install mlflow


In [ ]:
# Google auth — if 'credential propagation was unsuccessful', just re-run this cell
from google.colab import auth; auth.authenticate_user()
!gcloud config set project macrocosm-lewagon -q


In [ ]:
!mkdir -p /content/data
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v4.5/catalog_v4.parquet /content/data/catalog_v1.parquet
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v4.5/images_*.npy /content/data/
DATA_DIR = '/content/data'

import tensorflow as tf
print('GPUs =', tf.config.list_physical_devices('GPU'))


In [ ]:
try:
    from google.colab import userdata
    MLFLOW_TOKEN = userdata.get('MLFLOW_TOKEN')
except Exception as e:
    print('no MLFLOW_TOKEN secret ->', e); MLFLOW_TOKEN = None


In [ ]:
from tab_cnn import train_tab

MORPH = ['conc_r', 'log_petroR90_r']
REF = {'conc_r': (.713, .787), 'log_petroR90_r': (.828, .842)}   # (joint16, hard6)

hist_m, model_morph, (fm_m, fs_m) = train_tab(
    data_dir=DATA_DIR, crop=24, preproc='p99', features=MORPH,
    N=None, batch=256, lr=3e-4, epochs=50,
    run_name='tab-mdn-morph-v1', mlflow_token=MLFLOW_TOKEN)


## Eval + embedding


In [ ]:
import glob, re
import numpy as np, pandas as pd
import eval as ev, fusion as fu
from tab_cnn import tab_targets

cat, z_all, o2i, Y = tab_targets(DATA_DIR)
va = np.array([o2i[int(o)] for o in pd.read_csv('splits/v4-5-validate.csv').objid if int(o) in o2i])
SHARD = 6000
mm = {int(re.findall(r'images_(\d+)_', p)[0]) // SHARD: np.load(p, mmap_mode='r')
      for p in glob.glob(f'{DATA_DIR}/images_*.npy')}
X = np.empty((len(va), 24, 24, 5), np.float16)
for s in np.unique(va // SHARD):
    sel = np.where(va // SHARD == s)[0]; rr = va[sel] % SHARD; srt = np.argsort(rr)
    X[sel[srt]] = mm[int(s)][rr[srt]]
pp = ev.make_np_preprocess('p99')
raw = model_morph.predict(pp(np.asarray(X, 'float32')), batch_size=1024, verbose=0)
pi, mu = raw[..., :2], raw[..., 2:4]
pred = (pi * mu).sum(-1) * fs_m + fm_m
print(f"{'feature':<16} {'joint16':>8} {'hard6':>8} {'morph2':>8} {'d_vs_hard6':>11}")
for j, name in enumerate(MORPH):
    i = fu.TAB_FEATURES.index(name)
    msk = np.isfinite(Y[va, i])
    r2 = 1 - np.var(pred[msk, j] - Y[va, i][msk]) / np.var(Y[va, i][msk])
    print(f'{name:<16} {REF[name][0]:8.3f} {REF[name][1]:8.3f} {r2:8.3f} {r2-REF[name][1]:+11.3f}')
del X, raw


In [ ]:
import tensorflow as tf
embn = tf.keras.Model(model_morph.input, model_morph.get_layer('embedding').output)
paths = sorted(glob.glob(f'{DATA_DIR}/images_*.npy'), key=lambda p: int(re.findall(r'images_(\d+)_', p)[0]))
emb = np.concatenate([embn.predict(pp(np.asarray(np.load(p), 'float32')), batch_size=1024, verbose=0)
                      for p in paths])
np.save('/content/emb_tabmdn_morph_v1.npy', emb); print('emb', emb.shape)
!gcloud storage cp /content/emb_tabmdn_morph_v1.npy gs://macrocosm-lewagon/results/
